In [78]:
import pandas as pd

data = pd.read_csv("../data/datos_con_categoria.csv")

In [79]:
pd.set_option('display.max_columns', None)
import warnings; warnings.filterwarnings("ignore")

data.columns


Index(['seller_address', 'warranty', 'sub_status', 'condition',
       'seller_contact', 'deal_ids', 'base_price', 'shipping',
       'non_mercado_pago_payment_methods', 'seller_id', 'variations',
       'location', 'site_id', 'listing_type_id', 'price', 'attributes',
       'buying_mode', 'tags', 'listing_source', 'parent_item_id',
       'coverage_areas', 'category_id', 'descriptions', 'last_updated',
       'international_delivery_mode', 'pictures', 'id', 'official_store_id',
       'differential_pricing', 'accepts_mercadopago', 'original_price',
       'currency_id', 'thumbnail', 'title', 'automatic_relist', 'date_created',
       'secure_thumbnail', 'stop_time', 'status', 'video_id',
       'catalog_product_id', 'subtitle', 'initial_quantity', 'start_time',
       'permalink', 'geolocation', 'sold_quantity', 'available_quantity',
       'warranty_category'],
      dtype='object')

In [132]:

df_picked_cols = data[[
    "shipping",
    "non_mercado_pago_payment_methods",
    "price",
    "variations",
    "listing_type_id",
    "buying_mode",
    "tags",
    "official_store_id",
    "automatic_relist",
    "initial_quantity",
    "condition",
    "warranty_category",
    "sold_quantity",
    "seller_address",
    "title"

]]


In [81]:
df_picked_cols["shipping"].iloc[0]

"{'local_pick_up': True, 'methods': [], 'tags': [], 'free_shipping': False, 'mode': 'not_specified', 'dimensions': None}"

In [82]:
import ast

df_picked_cols["local_pick_up"] = df_picked_cols["shipping"].apply(lambda x: ast.literal_eval(x)["local_pick_up"] if pd.notnull(x) else None)
df_picked_cols["free_shipping"] = df_picked_cols["shipping"].apply(lambda x: ast.literal_eval(x)["free_shipping"] if pd.notnull(x) else None)
df_picked_cols["mode"] = df_picked_cols["shipping"].apply(lambda x: ast.literal_eval(x)["mode"] if pd.notnull(x) else None)

In [83]:
df_picked_cols.drop(columns=["shipping"], inplace=True)

In [84]:
df_picked_cols["non_mercado_pago_payment_methods"].iloc[0]

"[{'description': 'Transferencia bancaria', 'id': 'MLATB', 'type': 'G'}, {'description': 'Acordar con el comprador', 'id': 'MLAWC', 'type': 'G'}, {'description': 'Efectivo', 'id': 'MLAMO', 'type': 'G'}]"

In [85]:
import ast
from collections import defaultdict
descripcion_list = [
    'Visa Electron',
    'Mastercard Maestro',
    'Acordar con el comprador',
    'Transferencia bancaria',
    'Visa',
    'Contra reembolso',
    'MasterCard',
    'Cheque certificado',
    'Tarjeta de crédito',
    'Giro postal',
    'Diners',
    'Efectivo',
    'American Express',
    'MercadoPago'
 ]
def get_descriptions(row):
    try:
        items = ast.literal_eval(row) if pd.notnull(row) else []
        return set([item['description'] for item in items])
    except Exception:
        return set()
for desc in descripcion_list:
    df_picked_cols[desc] = df_picked_cols['non_mercado_pago_payment_methods'].apply(lambda x: 1 if desc in get_descriptions(x) else 0)

In [86]:
df_picked_cols.drop(columns=["non_mercado_pago_payment_methods"], inplace=True)

In [87]:
for row in df_picked_cols["variations"]:
    if row!='[]':
        print(row)
        break

[{'attribute_combinations': [{'value_id': '92012', 'name': 'Color Primario', 'value_name': 'Azul petróleo', 'id': '83000'}, {'value_id': '82034', 'name': 'Color Secundario', 'value_name': 'Amarillo', 'id': '73001'}, {'value_id': '141996', 'name': 'Talle', 'value_name': '8', 'id': '103000'}], 'seller_custom_field': None, 'picture_ids': ['472901-MLA20442937232_102015', '509801-MLA20442939057_102015', '650901-MLA20442940047_102015', '373901-MLA20442951026_102015', '422901-MLA20442950410_102015'], 'sold_quantity': 0, 'available_quantity': 1, 'id': 9742952789, 'price': 180}]


In [88]:
import ast
def get_items_variation(row):
    try:
        items = ast.literal_eval(row) if pd.notnull(row) else []
        return len(items)
    except Exception:
        return 0
def get_available_quantity_variation(row):
    try:
        items = ast.literal_eval(row) if pd.notnull(row) else []
        if not items:
            return 0
        total = sum(item.get('available_quantity', 0) for item in items)
        return total / len(items) if len(items) > 0 else 0
    except Exception:
        return 0
def get_proce_variation(row):
    try:
        items = ast.literal_eval(row) if pd.notnull(row) else []
        if not items:
            return 0
        total =  sum(item.get('price', 0) for item in items)
        return total / len(items) if len(items) > 0 else 0
    except Exception:
        return 0

df_picked_cols['items_variation'] = df_picked_cols['variations'].apply(get_items_variation)
df_picked_cols['available_quantity_variation'] = df_picked_cols['variations'].apply(get_available_quantity_variation)
#df_picked_cols['price_variation'] = df_picked_cols['variations'].apply(get_proce_variation)

In [89]:
df_picked_cols.drop(columns=["variations"], inplace=True)

items variations y available quantity deben ser 1 cuando no haya nada

In [90]:
listing_types = ['bronze', 'silver', 'free', 'gold_special', 'gold', 'gold_premium', 'gold_pro']
for lt in listing_types:
    df_picked_cols[lt] = (df_picked_cols['listing_type_id'] == lt).astype(int)

In [91]:
df_picked_cols.drop(columns=["listing_type_id"], inplace=True)

In [92]:
listing_types = ['buy_it_now', 'classified', 'auction']
for lt in listing_types:
    df_picked_cols[lt] = (df_picked_cols['buying_mode'] == lt).astype(int)

In [93]:
df_picked_cols.drop(columns=["buying_mode"], inplace=True)

In [94]:
df_picked_cols.head(3)


,price,tags,official_store_id,automatic_relist,initial_quantity,condition,warranty_category,sold_quantity,seller_address,title,local_pick_up,free_shipping,mode,Visa Electron,Mastercard Maestro,Acordar con el comprador,Transferencia bancaria,Visa,Contra reembolso,MasterCard,Cheque certificado,Tarjeta de crédito,Giro postal,Diners,Efectivo,American Express,MercadoPago,items_variation,available_quantity_variation,bronze,silver,free,gold_special,gold,gold_premium,gold_pro,buy_it_now,classified,auction
0,80.0,['dragged_bids_and_visits'],NaN,False,1,new,Sin garantía,0,"{'comment': '', 'longitude': -58.3986709, 'id'...",Auriculares Samsung Originales Manos Libres Ca...,True,False,not_specified,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0
1,2650.0,[],NaN,False,1,used,Garantía ilimitada / de por vida,0,"{'comment': '', 'longitude': -58.5059173, 'id'...",Cuchillo Daga Acero Carbón Casco Yelmo Solinge...,True,False,me2,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.0,0,1,0,0,0,0,0,1,0,0
2,60.0,['dragged_bids_and_visits'],NaN,False,1,used,Sin garantía,0,"{'comment': '', 'longitude': -58.4143948, 'id'...","Antigua Revista Billiken, N° 1826, Año 1954",True,False,me2,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0


In [95]:
tag_classes = [
    'dragged_bids_and_visits',
    'good_quality_thumbnail',
    'dragged_visits',
    'free_relist',
    'poor_quality_thumbnail'
    ]
def tag_dummies(tags):
    try:
        tags = ast.literal_eval(tags) if pd.notnull(tags) else []
    except Exception:
        tags = []
    if not isinstance(tags, list):
        return {tag: 0 for tag in tag_classes} | {'sin_tag': 1}
    if len(tags) == 0:
        return {tag: 0 for tag in tag_classes} | {'sin_tag': 1}
    d = {tag: int(tag in tags) for tag in tag_classes}
    d['sin_tag'] = 0
    return d
tag_df = df_picked_cols['tags'].apply(tag_dummies).apply(pd.Series)
df_picked_cols = pd.concat([df_picked_cols, tag_df], axis=1)

In [96]:
df_picked_cols.drop(columns=["tags"], inplace=True)

In [97]:
df_picked_cols['is_official_store'] = df_picked_cols['official_store_id'].notna().astype(int)

In [98]:
df_picked_cols.drop(columns=["official_store_id"], inplace=True)

In [99]:
# df_picked_cols[(df_picked_cols["warranty"].notnull()) & (df_picked_cols["condition"]=="used")][140:150]

In [100]:
# Revisa la distribución entre condition new y used con respecto a los vacios o nulos de la columna warranty
# df_picked_cols[(df_picked_cols['warranty'].isnull()) & (df_picked_cols["condition"]=="used")]



In [101]:
df_picked_cols["warranty_category"].unique()

array(['Sin garantía', 'Garantía ilimitada / de por vida',
       'Garantía de autenticidad / descripción',
       'Garantía oficial del fabricante / importador',
       'Garantía basada en reputación del vendedor',
       'Garantía con plazo explícito',
       'Garantía de inspección en el punto de entrega',
       'Garantía por defecto de fabricación',
       'Garantía de satisfacción / devolución de dinero'], dtype=object)

In [102]:
for col in ["automatic_relist", "local_pick_up", "free_shipping"]:
    df_picked_cols[col] = df_picked_cols[col].replace({False: 0, True: 1})

In [103]:

listing_types = ['custom', 'not_specified', 'me1', 'me2']
for lt in listing_types:
    df_picked_cols[lt] = (df_picked_cols['mode'] == lt).astype(int)

In [104]:
df_picked_cols.drop(columns=["mode"], inplace=True)

In [105]:
df_picked_cols.head(3)

,price,automatic_relist,initial_quantity,condition,warranty_category,sold_quantity,seller_address,title,local_pick_up,free_shipping,Visa Electron,Mastercard Maestro,Acordar con el comprador,Transferencia bancaria,Visa,Contra reembolso,MasterCard,Cheque certificado,Tarjeta de crédito,Giro postal,Diners,Efectivo,American Express,MercadoPago,items_variation,available_quantity_variation,bronze,silver,free,gold_special,gold,gold_premium,gold_pro,buy_it_now,classified,auction,dragged_bids_and_visits,good_quality_thumbnail,dragged_visits,free_relist,poor_quality_thumbnail,sin_tag,is_official_store,custom,not_specified,me1,me2
0,80.0,0,1,new,Sin garantía,0,"{'comment': '', 'longitude': -58.3986709, 'id'...",Auriculares Samsung Originales Manos Libres Ca...,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0
1,2650.0,0,1,used,Garantía ilimitada / de por vida,0,"{'comment': '', 'longitude': -58.5059173, 'id'...",Cuchillo Daga Acero Carbón Casco Yelmo Solinge...,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1
2,60.0,0,1,used,Sin garantía,0,"{'comment': '', 'longitude': -58.4143948, 'id'...","Antigua Revista Billiken, N° 1826, Año 1954",1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1


In [106]:
def categorizar_garantia(row):
    sin_garantia_vals = [
        'Sin garantía',
        'Garantía de autenticidad / descripción',
        'Garantía basada en reputación del vendedor'
    ]
    if row in sin_garantia_vals:
        return pd.Series({'sin_garantia': 1, 'garantia_especifica': 0})
    else:
        return pd.Series({'sin_garantia': 0, 'garantia_especifica': 1})

df_picked_cols[['sin_garantia', 'garantia_especifica']] = df_picked_cols['warranty_category'].apply(categorizar_garantia)

In [107]:

# listing_types = ['Sin garantía', 'Garantía ilimitada / de por vida',
#         'Garantía de autenticidad / descripción',
#         'Garantía oficial del fabricante / importador',
#         'Garantía basada en reputación del vendedor',
#         'Garantía con plazo explícito',
#         'Garantía de inspección en el punto de entrega',
#         'Garantía por defecto de fabricación',
#         'Garantía de satisfacción / devolución de dinero']
# for lt in listing_types:
#     df_picked_cols[lt] = (df_picked_cols['warranty_category'] == lt).astype(int)

In [108]:
df_picked_cols.drop(columns=["warranty_category"], inplace=True)

In [109]:
df_picked_cols['condition'] = df_picked_cols['condition'].replace({'new': 0, 'used': 1})

In [110]:
df_picked_cols

,price,automatic_relist,initial_quantity,condition,sold_quantity,seller_address,title,local_pick_up,free_shipping,Visa Electron,Mastercard Maestro,Acordar con el comprador,Transferencia bancaria,Visa,Contra reembolso,MasterCard,Cheque certificado,Tarjeta de crédito,Giro postal,Diners,Efectivo,American Express,MercadoPago,items_variation,available_quantity_variation,bronze,silver,free,gold_special,gold,gold_premium,gold_pro,buy_it_now,classified,auction,dragged_bids_and_visits,good_quality_thumbnail,dragged_visits,free_relist,poor_quality_thumbnail,sin_tag,is_official_store,custom,not_specified,me1,me2,sin_garantia,garantia_especifica
0,80.00,0,1,0,0,"{'comment': '', 'longitude': -58.3986709, 'id'...",Auriculares Samsung Originales Manos Libres Ca...,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0
1,2650.00,0,1,1,0,"{'comment': '', 'longitude': -58.5059173, 'id'...",Cuchillo Daga Acero Carbón Casco Yelmo Solinge...,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1
2,60.00,0,1,1,0,"{'comment': '', 'longitude': -58.4143948, 'id'...","Antigua Revista Billiken, N° 1826, Año 1954",1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0
3,580.00,0,1,0,0,"{'comment': '', 'longitude': -58.4929208, 'id'...",Alarma Guardtex Gx412 Seguridad Para El Automo...,1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0.0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0
4,30.00,0,1,1,0,"{'comment': '', 'longitude': -58.5495042, 'id'...",Serenata - Jennifer Blake,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89995,68.00,0,1,1,0,"{'comment': '', 'longitude': -58.5122086, 'id'...",El Fin De Las Libertades - Benegas Lynch (h) -...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0
89996,126.00,0,1,0,1,"{'comment': '', 'longitude': -58.3815931, 'id'...",Honda Wave Guardabarro Interior Trasero,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0
89997,300.00,0,1,0,0,"{'comment': '', 'longitude': -58.5899289, 'id'...",My Little Pony Completa Latino 4 Temporadas,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0
89998,696.58,0,100,0,0,"{'comment': '', 'longitude': -65.3128244, 'id'...",Accidente Cerebrovascular En La Infancia Y Ado...,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1


In [111]:
df_picked_cols["seller_state_name"] = df_picked_cols["seller_address"].apply(
    lambda x: eval(x).get("state", {}).get("name") if isinstance(eval(x), dict) else None
)
df_picked_cols.drop(columns=["seller_address"], inplace=True)

In [112]:
freq = df_picked_cols['seller_state_name'].value_counts(normalize=True)
cumfreq = freq.cumsum()



top_states = ['Capital Federal', 'Buenos Aires', 'Santa Fe', 'Córdoba', 'Mendoza']


df_picked_cols['state_grouped'] = df_picked_cols['seller_state_name'].where(
    df_picked_cols['seller_state_name'].isin(top_states),
    other='Otros'
)

df_picked_cols = pd.get_dummies(df_picked_cols, columns=['state_grouped'], drop_first=True)

In [113]:
df_picked_cols.drop(columns=["seller_state_name"], inplace=True)

In [114]:
bool_cols = df_picked_cols.select_dtypes(include=['bool']).columns
for col in bool_cols:
    df_picked_cols[col] = df_picked_cols[col].astype(int)

In [115]:
#df_picked_cols.drop(columns=["price"], inplace = True)

In [116]:
# import pandas as pd

# # Selecciona la columna base_price sin nulos
# base_price = df_picked_cols["price"].dropna()

# # Define los límites manualmente según tu resultado
# bins = [0, 73, 165, 375, 1150, base_price.max() + 1]
# labels = [
#     "0-73",
#     "73-165",
#     "165-375",
#     "375-1150",
#     "1150+"
# ]

# # Crea la columna de categorías
# df_picked_cols["base_price_category"] = pd.cut(df_picked_cols["price"], bins=bins, labels=labels, include_lowest=True)


In [117]:
# df_picked_cols.drop(columns=["price"], inplace=True)

In [118]:
# listing_types = [
#     "0-73",
#     "73-165",
#     "165-375",
#     "375-1150",
#     "1150+"
# ]
# for lt in listing_types:
#     df_picked_cols[lt] = (df_picked_cols['base_price_category'] == lt).astype(int)

In [119]:
# df_picked_cols.drop(columns=["base_price_category"], inplace=True)

In [120]:
# df_picked_cols.drop(columns=["sin_garantia", "garantia_especifica"], inplace=True)

In [133]:
df_picked_cols.head(3)

,shipping,non_mercado_pago_payment_methods,price,variations,listing_type_id,buying_mode,tags,official_store_id,automatic_relist,initial_quantity,condition,warranty_category,sold_quantity,seller_address,title
0,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",80.0,[],bronze,buy_it_now,['dragged_bids_and_visits'],NaN,False,1,new,Sin garantía,0,"{'comment': '', 'longitude': -58.3986709, 'id'...",Auriculares Samsung Originales Manos Libres Ca...
1,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",2650.0,[],silver,buy_it_now,[],NaN,False,1,used,Garantía ilimitada / de por vida,0,"{'comment': '', 'longitude': -58.5059173, 'id'...",Cuchillo Daga Acero Carbón Casco Yelmo Solinge...
2,"{'local_pick_up': True, 'methods': [], 'tags':...","[{'description': 'Transferencia bancaria', 'id...",60.0,[],bronze,buy_it_now,['dragged_bids_and_visits'],NaN,False,1,used,Sin garantía,0,"{'comment': '', 'longitude': -58.4143948, 'id'...","Antigua Revista Billiken, N° 1826, Año 1954"


In [122]:
# Balanceo de clases para la variable 'condition' (target)
# from sklearn.utils import resample

# # Separar clases
# df_majority = df_picked_cols[df_picked_cols['condition'] == 0]
# df_minority = df_picked_cols[df_picked_cols['condition'] == 1]

# # Re-muestreo de la clase minoritaria
# df_minority_upsampled = resample(
#     df_minority,
#     replace=True,
#     n_samples=len(df_majority),
#     random_state=42
# )

# # Unir clases balanceadas
# df_balanced = pd.concat([df_majority, df_minority_upsampled])

# # Mezclar el dataset
# df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# # Verifica el balanceo
# print(df_balanced['condition'].value_counts())

In [123]:
# df_res.to_csv("data_clean.csv", index=False)

In [124]:
from repositories.title_predictor import TitleProcessor

processor = TitleProcessor(
    model_path='../data/modelo_final.joblib',
    scaler_path='../data/modelo_scaler.joblib',
    encoder_path='../data/modelo_encoder.joblib'
)



2025-07-12 11:56:48,469 - INFO - Inicializando ModeloHibridoOptimizado


In [125]:
categorias, probabilidades = processor.predict_titles(df_picked_cols["title"].tolist())
df_picked_cols["categoria_predicha"] = categorias
df_picked_cols["probabilidad_new"] = probabilidades[:, 0]
df_picked_cols["probabilidad_used"] = probabilidades[:, 1]

2025-07-12 11:56:48,496 - INFO - Preparando datos para 90000 registros...


Preparando datos para 90000 registros...


2025-07-12 11:56:48,830 - INFO - Use pytorch device_name: cuda:0
2025-07-12 11:56:48,831 - INFO - Load pretrained SentenceTransformer: paraphrase-multilingual-MiniLM-L12-v2


Cargando modelo de embeddings...


Batches: 100%|██████████| 2813/2813 [00:35<00:00, 78.84it/s]


Embeddings shape: (90000, 384)
Extrayendo features avanzadas...
Features engineeradas: 57 features


2025-07-12 11:57:35,668 - INFO - Datos preparados exitosamente: (90000, 441)


Dataset final: (90000, 441)


In [126]:
df_picked_cols.head(3)

,price,automatic_relist,initial_quantity,condition,sold_quantity,title,local_pick_up,free_shipping,Visa Electron,Mastercard Maestro,Acordar con el comprador,Transferencia bancaria,Visa,Contra reembolso,MasterCard,Cheque certificado,Tarjeta de crédito,Giro postal,Diners,Efectivo,American Express,MercadoPago,items_variation,available_quantity_variation,bronze,silver,free,gold_special,gold,gold_premium,gold_pro,buy_it_now,classified,auction,dragged_bids_and_visits,good_quality_thumbnail,dragged_visits,free_relist,poor_quality_thumbnail,sin_tag,is_official_store,custom,not_specified,me1,me2,sin_garantia,garantia_especifica,state_grouped_Capital Federal,state_grouped_Córdoba,state_grouped_Mendoza,state_grouped_Otros,state_grouped_Santa Fe,categoria_predicha,probabilidad_new,probabilidad_used
0,80.0,0,1,0,0,Auriculares Samsung Originales Manos Libres Ca...,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,new,0.812019,0.187981
1,2650.0,0,1,1,0,Cuchillo Daga Acero Carbón Casco Yelmo Solinge...,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0.0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,1,0,0,0,0,new,0.662169,0.337831
2,60.0,0,1,1,0,"Antigua Revista Billiken, N° 1826, Año 1954",1,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0.0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,used,0.001765,0.998235


In [127]:
df_picked_cols.drop(columns=["categoria_predicha"], inplace=True)

In [128]:
df_picked_cols.drop(columns=["title"], inplace=True)

In [129]:
from imblearn.over_sampling import SMOTENC


X = df_picked_cols.drop('condition', axis=1)
y = df_picked_cols['condition']

categorical_features = [
    i for i, col in enumerate(X.columns)
    if X[col].nunique() == 2
]

smote_nc = SMOTENC(categorical_features=categorical_features, random_state=42)

X_res, y_res = smote_nc.fit_resample(X, y)

df_res = pd.concat([
    pd.DataFrame(X_res, columns=X.columns),
    pd.Series(y_res, name='condition')
], axis=1)

print("Original:", df_picked_cols.shape, "\n", y.value_counts(), "\n")
print("Balanceado:", df_res.shape, "\n", df_res['condition'].value_counts())

Original: (90000, 53) 
 condition
0    48352
1    41648
Name: count, dtype: int64 

Balanceado: (96704, 53) 
 condition
0    48352
1    48352
Name: count, dtype: int64


In [130]:
df_picked_cols.to_csv("../data/data_clean.csv", index=False)